In [35]:

# ``pip install sentence-transformers``
# This lightweight 2025-style pipeline embeds messages and tags them without expensive prompts


In [36]:

from typing import List

from sentence_transformers import SentenceTransformer, util

TOPIC_LABELS = [
  "national identity",
  "identity & exclusion",
  "conspiracy narratives",
  "street action",
  "doctrine",
  "electoral politics",
];


topic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)
topic_embeddings = topic_model.encode(
    TOPIC_LABELS, convert_to_tensor=True, normalize_embeddings=True
)

def predict_topics(text: str, threshold: float = 0.32, top_k: int = 3) -> List[str]:
    if not text:
        return []
    emb = topic_model.encode(text, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.dot_score(emb, topic_embeddings)[0]
    scored = sorted(
        [(float(score), label) for score, label in zip(scores, TOPIC_LABELS)],
        reverse=True,
        key=lambda pair: pair[0],
    )
    selected = [label for score, label in scored if score >= threshold]
    if not selected and scored:
        selected = [scored[0][1]]
    return selected[:top_k]


In [37]:

import pandas as pd
from pathlib import Path

sample_size = 1000
topic_threshold = 0.32
max_topics = 4

base_dir = Path("data")
message_paths = sorted(base_dir.glob("*/message_nodes.csv"))
samples = []

for message_path in message_paths:
    df = pd.read_csv(message_path)
    df = df[df['text'].notna()].copy()
    df['text'] = df['text'].astype(str)
    if df.empty:
        continue
    if len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42)
    df['source_file'] = str(message_path)
    samples.append(df)

sample = pd.concat(samples).reset_index(drop=True) if samples else pd.DataFrame()


In [38]:
from tqdm.auto import tqdm
import time

if sample.empty:
    print("No samples available to tag.")
else:
    tqdm.pandas()
    start = time.perf_counter()
    sample['topics'] = sample['text'].progress_apply(
        lambda text: predict_topics(text, threshold=topic_threshold, top_k=max_topics)
    )
    end = time.perf_counter()

    sample['tagging_error'] = None
    print(f"Tagged {len(sample)} messages in {end - start:.1f} seconds")


  0%|          | 0/4000 [00:00<?, ?it/s]

Tagged 4000 messages in 32.4 seconds


In [39]:
from pathlib import Path
import json
from tqdm.auto import tqdm

# mirror the crawler’s tag-updating loop but using our fast embed classifier
base_dir = Path("data")
graph_paths = sorted(base_dir.glob("*/graph.json"))
min_text_length = 15

for graph_path in graph_paths:
    graph = json.loads(graph_path.read_text(encoding="utf-8"))
    messages = graph.get("messages", [])
    if not messages:
        print(f"skip {graph_path.parent.name}: no messages")
        continue

    desc = graph_path.parent.name
    for msg in tqdm(messages, desc=f"topics {desc}", unit="msg"):
        text = (msg.get("text") or "").strip()
        msg["topics"] = (
            predict_topics(text, threshold=topic_threshold, top_k=max_topics)
            if len(text) >= min_text_length
            else []
        )

    graph_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")
    # print(f"Updated {desc} ({len(messages)} messages)")

topics afdjugendbw:   0%|          | 0/31704 [00:00<?, ?msg/s]

topics generationidentitaire:   0%|          | 0/20016 [00:00<?, ?msg/s]

topics jungenationalisten:   0%|          | 0/30254 [00:00<?, ?msg/s]

topics tricoloredelsangueitalico:   0%|          | 0/35936 [00:00<?, ?msg/s]

In [40]:

from pathlib import Path
import pandas as pd

min_text_length = 15
base_dir = Path("data")
message_paths = sorted(base_dir.glob("*/message_nodes.csv"))
total = 0

for message_path in message_paths:
    df = pd.read_csv(message_path)
    df['text'] = df['text'].fillna("").astype(str)
    df['topics'] = df['text'].apply(
        lambda text: (
            predict_topics(text, threshold=topic_threshold, top_k=max_topics)
            if len(text.strip()) >= min_text_length
            else []
        )
    )
    # out_path = message_path.with_name("message_nodes_tagged.csv")
    out_path = message_path.with_name("message_nodes.csv")

    df.to_csv(out_path, index=False)
    total += len(df)
    print(f"Updated {message_path.parent.name}: {len(df)} rows -> {out_path.name}")
print(f"Completed updating {len(message_paths)} message files ({total} rows) to message_nodes_tagged.csv")


Updated afdjugendbw: 31704 rows -> message_nodes.csv
Updated generationidentitaire: 20016 rows -> message_nodes.csv
Updated jungenationalisten: 30254 rows -> message_nodes.csv
Updated tricoloredelsangueitalico: 35936 rows -> message_nodes.csv
Completed updating 4 message files (117910 rows) to message_nodes_tagged.csv


In [41]:

sample.head()


,id,chat,message_id,date,url,views,reaction_count,reaction_breakdown,text,sender_id,source_file,topics,tagging_error
0,kulturfestung:210,kulturfestung,210,2021-08-23T07:33:11+00:00,https://t.me/kulturfestung/210,447,0,{},Vortrag Blockchain Technologie am 21.8.21 in d...,-1.001267e+12,data/afdjugendbw/message_nodes.csv,[doctrine],None
1,rtvooe:3291,rtvooe,3291,2025-11-24T10:16:16+00:00,https://t.me/rtvooe/3291,1791,106,"{""👍"": 85, ""👏"": 12, ""❤"": 7, ""🔥"": 1, ""😁"": 1}",FPÖ überholt ÖVP in Oberösterreich – Alarm in ...,-1.001295e+12,data/afdjugendbw/message_nodes.csv,[street action],None
2,ein_prozent:3197,ein_prozent,3197,2024-09-20T10:03:53+00:00,https://t.me/ein_prozent/3197,7280,99,"{""👏"": 68, ""👍"": 19, ""🔥"": 7, ""💩"": 5}","Wir sind live und freuen uns, wenn ihr den Lin...",-1.001414e+12,data/afdjugendbw/message_nodes.csv,[doctrine],None
3,WolfPMSoffiziell:1129,WolfPMSoffiziell,1129,2024-03-09T14:56:26+00:00,https://t.me/WolfPMSoffiziell/1129,1623,0,{},🌅Arno « GOAT» Breker🏴‍☠️,-1.001447e+12,data/afdjugendbw/message_nodes.csv,[national identity],None
4,juergenpohlafd:1109,juergenpohlafd,1109,2025-09-29T20:51:26+00:00,https://t.me/juergenpohlafd/1109,266,10,"{""❤"": 8, ""💯"": 2}",Der alte Fehler in neuen Kleidern - der neuste...,-1.001805e+12,data/afdjugendbw/message_nodes.csv,[national identity],None
